In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

PyTorch: 2.10.0+cu128
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [5]:
!nvidia-smi

Fri Sep 11 12:17:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import os

In [7]:
import pandas as pd

BASE = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset"

nutrition = pd.read_csv(f"{BASE}/dish_nutrition_values.csv")

print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein']


,dish_id,calories,mass,fat,carb,protein
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970
1,dish_1562688426,137.569992,88.0,8.256000,5.190000,10.297000
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000


In [8]:
import os

nutrition["image_path"] = nutrition["dish_id"].apply(
    lambda x: f"{BASE}/imagery/realsense_overhead/{x}/rgb.png"
)

# Chỉ giữ món thực sự có ảnh RGB
nutrition = nutrition[
    nutrition["image_path"].apply(os.path.exists)
].reset_index(drop=True)

print("Số món có ảnh:", len(nutrition))

display(
    nutrition[
        ["dish_id", "image_path", "calories"]
    ].head()
)

Số món có ảnh: 3244


,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


In [9]:
print(nutrition.columns.tolist())
display(nutrition.head())

['dish_id', 'calories', 'mass', 'fat', 'carb', 'protein', 'image_path']


,dish_id,calories,mass,fat,carb,protein,image_path
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,/kaggle/input/datasets/gillesokhin/nutrition5k...
1,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,/kaggle/input/datasets/gillesokhin/nutrition5k...
2,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,/kaggle/input/datasets/gillesokhin/nutrition5k...
3,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,/kaggle/input/datasets/gillesokhin/nutrition5k...
4,dish_1558372433,74.360001,143.0,0.286000,0.429000,20.020000,/kaggle/input/datasets/gillesokhin/nutrition5k...


In [10]:
df = nutrition[
    ["dish_id", "image_path", "calories"]
].copy()

display(df.head())
print("Tổng số mẫu:", len(df))

,dish_id,image_path,calories
0,dish_1561662216,/kaggle/input/datasets/gillesokhin/nutrition5k...,300.794281
1,dish_1561662054,/kaggle/input/datasets/gillesokhin/nutrition5k...,419.438782
2,dish_1562008979,/kaggle/input/datasets/gillesokhin/nutrition5k...,382.936646
3,dish_1560455030,/kaggle/input/datasets/gillesokhin/nutrition5k...,20.590000
4,dish_1558372433,/kaggle/input/datasets/gillesokhin/nutrition5k...,74.360001


Tổng số mẫu: 3244


In [11]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2270
Val: 487
Test: 487


In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class NutritionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        calories = torch.tensor(row["calories"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, calories

In [13]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [14]:
train_dataset = NutritionDataset(train_df, train_transform)
val_dataset = NutritionDataset(val_df, val_transform)
test_dataset = NutritionDataset(test_df, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

2270 487 487


In [15]:
images, calories = next(iter(train_loader))

print("Images:", images.shape)
print("Calories:", calories.shape)
print("Ví dụ calories:", calories[:5])

Images: torch.Size([64, 3, 224, 224])
Calories: torch.Size([64])
Ví dụ calories: tensor([322.0020,  38.0700, 111.9600, 351.4055, 696.3635])


In [16]:
# Lưu split cố định để mọi experiment sau dùng cùng dữ liệu
train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)

print("Saved fixed splits:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Saved fixed splits:
Train: 2270
Validation: 487
Test: 487


In [17]:
# Thống kê mô tả target calories
def calorie_stats(dataframe, split_name):
    return {
        "split": split_name,
        "n_samples": len(dataframe),
        "min_kcal": dataframe["calories"].min(),
        "max_kcal": dataframe["calories"].max(),
        "mean_kcal": dataframe["calories"].mean(),
        "std_kcal": dataframe["calories"].std(),
        "median_kcal": dataframe["calories"].median(),
    }

stats_df = pd.DataFrame([
    calorie_stats(train_df, "Train"),
    calorie_stats(val_df, "Validation"),
    calorie_stats(test_df, "Test"),
    calorie_stats(pd.concat([train_df, val_df, test_df]), "Total"),
])

display(stats_df.round(2))
stats_df.to_csv("/kaggle/working/dataset_calorie_statistics.csv", index=False)


,split,n_samples,min_kcal,max_kcal,mean_kcal,std_kcal,median_kcal
0,Train,2270,0.00,3943.33,253.31,221.40,207.40
1,Validation,487,2.30,1324.08,256.67,212.09,212.16
2,Test,487,1.15,1264.55,266.35,218.93,234.58
3,Total,3244,0.00,3943.33,255.77,219.64,210.32


# EXPERIMENTS

Phần chuẩn bị dữ liệu phía trên được giữ cố định.  
Thêm các experiment mới từ đây trở xuống để bảo đảm tất cả thí nghiệm dùng cùng dataset split.


In [18]:
# ============================================================
# EXPERIMENT 4 - RGB-D FUSION
# Step 1: Add depth paths
# ============================================================

import os

def add_depth_path(dataframe):
    df_out = dataframe.copy()

    df_out["depth_path"] = (
        df_out["image_path"]
        .str.replace("rgb.png", "depth_raw.png", regex=False)
    )

    return df_out


train_df = add_depth_path(train_df)
val_df   = add_depth_path(val_df)
test_df  = add_depth_path(test_df)


def check_depth_coverage(dataframe, name):
    exists = dataframe["depth_path"].apply(os.path.exists)

    print(
        f"{name}: "
        f"{exists.sum()}/{len(dataframe)} depth files "
        f"({exists.mean() * 100:.2f}%)"
    )

    return exists


train_depth_exists = check_depth_coverage(train_df, "Train")
val_depth_exists   = check_depth_coverage(val_df, "Validation")
test_depth_exists  = check_depth_coverage(test_df, "Test")

Train: 2270/2270 depth files (100.00%)
Validation: 487/487 depth files (100.00%)
Test: 486/487 depth files (99.79%)
